In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import gc
import os

In [2]:
dataset_dir = "dataset"

## Feature engineering

The `agg_numeric_columns` function creates a new DataFrame with aggregated numeric columns of the given DataFrame.

In [4]:
def agg_numeric_columns(df, groupby_columns, metrics=["count", "mean", "sum", "min", "max"], exclude_columns=None):
    exclude_columns = exclude_columns or []

    # Find all numeric columns that are not used for grouping
    numeric_columns = df.select_dtypes(include="number").columns.tolist()
    numeric_columns = [c for c in numeric_columns if c not in groupby_columns and c not in exclude_columns]

    if not numeric_columns:
        return df[groupby_columns].drop_duplicates().reset_index(drop=True)
    
    # Compute the aggregations and return a multi-index DataFrame
    multi_index_agg = df.groupby(groupby_columns)[numeric_columns].agg(metrics)

    # Flatten the DataFrame
    agg_df = pd.DataFrame({
        f"{column}_{agg}": multi_index_agg[(column, agg)]
        for column, agg in multi_index_agg.columns
    }).reset_index()

    # Only "count"/"sum" of an empty group are 0, for "mean"/"min"/"max"
    # a missing value means "no data", so leave it as NaN to avoid misleading the model.
    fill_zero_columns = [
        f"{column}_{agg}"
        for column, agg in multi_index_agg.columns
        if agg in ("count", "sum")
    ]
    agg_df[fill_zero_columns] = agg_df[fill_zero_columns].fillna(0)

    return agg_df

The `agg_categorical_columns` function creates a new DataFrame with aggregated categorical
columns of the given DataFrame.

In [5]:
def agg_categorical_columns(df, groupby_columns, exclude_columns=None):
    exclude_columns = exclude_columns or []
    categorical_columns = [
        c for c in df.select_dtypes(include="str").columns
        if c not in groupby_columns and c not in exclude_columns
    ]

    if not categorical_columns:
        return df[groupby_columns].drop_duplicates().reset_index(drop=True)
    
    one_hot_df = pd.get_dummies(df[groupby_columns + categorical_columns])
    multi_index_agg = one_hot_df.groupby(groupby_columns).agg(["sum", "mean"])
    multi_index_agg = multi_index_agg.rename(columns={"sum": "count", "mean": "count_norm"}, level=1)
    
    agg_df = pd.DataFrame({
        f"{column}_{agg}": multi_index_agg[(column, agg)]
        for column, agg in multi_index_agg.columns
    }).reset_index()
    
    return agg_df

A method that aggregates both numeric and categorical features of a dataframe.

In [6]:
def agg_columns(df, groupby_columns, exclude_columns=None, prefix=None):
    agg_numeric_df = agg_numeric_columns(df=df, groupby_columns=groupby_columns, exclude_columns=exclude_columns)
    agg_categorical_df = agg_categorical_columns(df=df, groupby_columns=groupby_columns, exclude_columns=exclude_columns)

    agg_df = pd.merge(agg_numeric_df, agg_categorical_df, on=groupby_columns)

    if prefix:
        agg_df = agg_df.rename(columns={
            column: f"{prefix}_{column}"
            for column in agg_df.columns
            if column not in groupby_columns
        })

    return agg_df

### The bureau tables

Load the bureau tables.

In [7]:
bureau_balance_df = pd.read_csv(os.path.join(dataset_dir, "bureau_balance.csv"))
bureau_df = pd.read_csv(os.path.join(dataset_dir, "bureau.csv"))

Aggregate the *bureau_balance* table and merge it with the *bureau* table.

In [8]:
agg_bureau_balance_df = agg_columns(bureau_balance_df, groupby_columns=["SK_ID_BUREAU"], prefix="BAL")
merged_bureau_bureau_balance_df = pd.merge(bureau_df, agg_bureau_balance_df, on="SK_ID_BUREAU", how="left")

Free memory.

In [9]:
del bureau_balance_df, bureau_df, agg_bureau_balance_df
gc.collect()

8

Aggregate the merged table and merge it with the main *applications* table.

In [10]:
merged_bureau_df = agg_columns(
    merged_bureau_bureau_balance_df,
    groupby_columns=["SK_ID_CURR"],
    exclude_columns=["SK_ID_BUREAU"],
    prefix="BUREAU"
)

Free memory.

In [11]:
del merged_bureau_bureau_balance_df
gc.collect()

0

In [12]:
merged_bureau_df

,SK_ID_CURR,BUREAU_DAYS_CREDIT_count,BUREAU_DAYS_CREDIT_mean,BUREAU_DAYS_CREDIT_sum,BUREAU_DAYS_CREDIT_min,BUREAU_DAYS_CREDIT_max,BUREAU_CREDIT_DAY_OVERDUE_count,BUREAU_CREDIT_DAY_OVERDUE_mean,BUREAU_CREDIT_DAY_OVERDUE_sum,BUREAU_CREDIT_DAY_OVERDUE_min,...,BUREAU_CREDIT_TYPE_Microloan_count,BUREAU_CREDIT_TYPE_Microloan_count_norm,BUREAU_CREDIT_TYPE_Mobile operator loan_count,BUREAU_CREDIT_TYPE_Mobile operator loan_count_norm,BUREAU_CREDIT_TYPE_Mortgage_count,BUREAU_CREDIT_TYPE_Mortgage_count_norm,BUREAU_CREDIT_TYPE_Real estate loan_count,BUREAU_CREDIT_TYPE_Real estate loan_count_norm,BUREAU_CREDIT_TYPE_Unknown type of loan_count,BUREAU_CREDIT_TYPE_Unknown type of loan_count_norm
0,100001,7,-735.000000,-5145,-1572,-49,7,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
1,100002,8,-874.000000,-6992,-1437,-103,8,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,100003,4,-1400.750000,-5603,-2586,-606,4,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
3,100004,2,-867.000000,-1734,-1326,-408,2,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
4,100005,3,-190.666667,-572,-373,-62,3,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305806,456249,13,-1667.076923,-21672,-2713,-483,13,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
305807,456250,3,-862.000000,-2586,-1002,-760,3,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
305808,456253,4,-867.500000,-3470,-919,-713,4,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
305809,456254,1,-1104.000000,-1104,-1104,-1104,1,0.0,0,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


### The Home Credit tables

Load the Home Credit tables.

In [13]:
credit_card_balance_df = pd.read_csv(os.path.join(dataset_dir, "credit_card_balance.csv"))
installments_payments_df = pd.read_csv(os.path.join(dataset_dir, "installments_payments.csv"))
POS_CASH_balance_df = pd.read_csv(os.path.join(dataset_dir, "POS_CASH_balance.csv"))
previous_application_df = pd.read_csv(os.path.join(dataset_dir, "previous_application.csv"))

Aggregate the *POS_CASH_balance* table and merge it with the *previous_application* table

In [14]:
agg_POS_CASH_balance_df = agg_columns(
    POS_CASH_balance_df,
    groupby_columns=["SK_ID_PREV"],
    exclude_columns=["SK_ID_CURR"],
    prefix="POS"
)
merged_previous_application_df = pd.merge(previous_application_df, agg_POS_CASH_balance_df, on="SK_ID_PREV", how="left")

Free memory.

In [15]:
del POS_CASH_balance_df, previous_application_df, agg_POS_CASH_balance_df
gc.collect()

0

Aggregate the *installments_payment* table and merge it with the *previous_application* table

In [16]:
agg_installments_payments_df = agg_columns(
    installments_payments_df,
    groupby_columns=["SK_ID_PREV"],
    exclude_columns=["SK_ID_CURR"],
    prefix="INSTALL"
)
merged_previous_application_df = pd.merge(merged_previous_application_df, agg_installments_payments_df, on="SK_ID_PREV", how="left")

Free memory.

In [17]:
del installments_payments_df, agg_installments_payments_df
gc.collect()

0

Aggregate the *credit_card_balance* table and merge it with the *previous_application* table

In [18]:
agg_credit_card_balance_df = agg_columns(
    credit_card_balance_df,
    groupby_columns=["SK_ID_PREV"],
    exclude_columns=["SK_ID_CURR"],
    prefix="CC"
)
merged_previous_application_df = pd.merge(merged_previous_application_df, agg_credit_card_balance_df, on="SK_ID_PREV", how="left")

Free memory.

In [19]:
del credit_card_balance_df, agg_credit_card_balance_df
gc.collect()

0

Aggregate the merged *previous_application* table.

In [20]:
merged_home_credit_df = agg_columns(
    merged_previous_application_df,
    groupby_columns=["SK_ID_CURR"],
    exclude_columns=["SK_ID_PREV"],
    prefix="PREV"
)

Free memory.

In [21]:
del merged_previous_application_df
gc.collect()

0

Merge the aggregated bureau the Home Credit dataframes.

In [22]:
to_merge_df = pd.merge(merged_bureau_df, merged_home_credit_df, on="SK_ID_CURR", how="outer")

Free memory.

In [23]:
del merged_bureau_df, merged_home_credit_df
gc.collect()

0

In [24]:
to_merge_df

,SK_ID_CURR,BUREAU_DAYS_CREDIT_count,BUREAU_DAYS_CREDIT_mean,BUREAU_DAYS_CREDIT_sum,BUREAU_DAYS_CREDIT_min,BUREAU_DAYS_CREDIT_max,BUREAU_CREDIT_DAY_OVERDUE_count,BUREAU_CREDIT_DAY_OVERDUE_mean,BUREAU_CREDIT_DAY_OVERDUE_sum,BUREAU_CREDIT_DAY_OVERDUE_min,...,PREV_PRODUCT_COMBINATION_POS industry without interest_count,PREV_PRODUCT_COMBINATION_POS industry without interest_count_norm,PREV_PRODUCT_COMBINATION_POS mobile with interest_count,PREV_PRODUCT_COMBINATION_POS mobile with interest_count_norm,PREV_PRODUCT_COMBINATION_POS mobile without interest_count,PREV_PRODUCT_COMBINATION_POS mobile without interest_count_norm,PREV_PRODUCT_COMBINATION_POS other with interest_count,PREV_PRODUCT_COMBINATION_POS other with interest_count_norm,PREV_PRODUCT_COMBINATION_POS others without interest_count,PREV_PRODUCT_COMBINATION_POS others without interest_count_norm
0,100001,7.0,-735.000000,-5145.0,-1572.0,-49.0,7.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0
1,100002,8.0,-874.000000,-6992.0,-1437.0,-103.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.0,0.0,1.0,1.0,0.0,0.0
2,100003,4.0,-1400.750000,-5603.0,-2586.0,-606.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
3,100004,2.0,-867.000000,-1734.0,-1326.0,-408.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.00,1.0,1.0,0.0,0.0,0.0,0.0
4,100005,3.0,-190.666667,-572.0,-373.0,-62.0,3.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353572,456251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0
353573,456252,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
353574,456253,4.0,-867.500000,-3470.0,-919.0,-713.0,4.0,0.0,0.0,0.0,...,0.0,0.0,2.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0
353575,456254,1.0,-1104.000000,-1104.0,-1104.0,-1104.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0


## Data preprocessing

Load the main *application_train.csv* table and merge it with the aggregated data table.

In [25]:
applications_df = pd.read_csv(os.path.join(dataset_dir, "application_train.csv"))
data_df = pd.merge(applications_df, to_merge_df, on="SK_ID_CURR", how="left")

Encode categorical features with one-hot encoding.

In [26]:
data_df = pd.get_dummies(data_df)

Free memory.

In [27]:
del applications_df, to_merge_df
gc.collect()

0

### Saving data to disk

In [28]:
data_df.to_csv(os.path.join(dataset_dir, "merged_data.csv"), index=False)